In [ ]:
# Cell 1: Install dependencies
!pip install pydub beautifulsoup4 requests
!apt-get -qq install -y ffmpeg

In [ ]:
# Cell 2: Clone repo and set configuration
!git clone https://github.com/bibleman-stan/readers-bofm.git
%cd readers-bofm

# ── Configuration ──
ELEVENLABS_API_KEY = "YOUR_ELEVENLABS_API_KEY_HERE"
VOICE_ID = "GBc0W7zMgpvEFBEHSpqS"   # Hector the Protector
MODEL_ID = "eleven_multilingual_v2"    # Best for accented English

LINE_PAUSE_MS = 180     # 180ms between sense-lines
VERSE_PAUSE_MS = 500    # 500ms between verses

print("Config set!")
print(f"  Voice ID: {VOICE_ID}")
print(f"  Model: {MODEL_ID}")

In [ ]:
# Cell 3: Core functions — HTML parsing + ElevenLabs TTS + Line Caching
import re
import json
import time
import hashlib
import requests
import io
import os
from pathlib import Path
from bs4 import BeautifulSoup, NavigableString
from pydub import AudioSegment

# ── Voice Settings (tune these to taste) ──
VOICE_SETTINGS = {
    "stability": 0.50,
    "similarity_boost": 0.75,
    "style": 0.35,
    "use_speaker_boost": True,
}

# ── Cache directory for per-line audio ──
CACHE_DIR = Path('audio_cache')
CACHE_DIR.mkdir(exist_ok=True)

def cache_key(text, voice_id, model_id, settings):
    """Generate a deterministic cache key from text + voice config."""
    blob = json.dumps({
        "text": text,
        "voice_id": voice_id,
        "model_id": model_id,
        "settings": settings,
    }, sort_keys=True)
    return hashlib.sha256(blob.encode()).hexdigest()[:16]

def get_cached_audio(key):
    """Return AudioSegment from cache if it exists, else None."""
    path = CACHE_DIR / f"{key}.mp3"
    if path.exists():
        return AudioSegment.from_mp3(path)
    return None

def save_to_cache(key, audio_seg):
    """Save an AudioSegment to the cache."""
    path = CACHE_DIR / f"{key}.mp3"
    audio_seg.export(str(path), format="mp3", bitrate="128k")


# ── HTML Parsing (same logic as generate_audio.py) ──

def get_text_from_element(el, use_modern=False):
    """Extract readable text, using modern word forms."""
    clone = BeautifulSoup(str(el), 'html.parser')
    if use_modern:
        for swap in clone.find_all(class_=['swap', 'swap-quiet']):
            mod = swap.get('data-mod')
            if mod:
                swap.string = mod
    for remove in clone.find_all(class_=['verse-num', 'parry-label-spacer', 'parry-label']):
        remove.decompose()
    text = clone.get_text()
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def parse_chapter(html_path, book_id, chapter_num):
    """Parse a chapter into speakable items with line indices."""
    with open(html_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')

    chapter_id = f'ch-{book_id}-{chapter_num}'
    chapter_div = soup.find(id=chapter_id)
    if not chapter_div:
        print(f'  Warning: chapter {chapter_id} not found')
        return []

    items = []
    line_idx = 0

    for child in chapter_div.children:
        if isinstance(child, NavigableString):
            continue
        classes = child.get('class', [])

        # Pericope header — skip audio but count in line_index for DOM alignment
        if 'pericope-header' in classes:
            line_idx += 1
            continue

        if 'verse' in classes:
            verse_num_el = child.find(class_='verse-num')
            vn = verse_num_el.get_text().strip() if verse_num_el else ''
            sense_lines = [
                el for el in child.find_all('span', recursive=False)
                if el.get('class') == ['line']
            ]
            for line_el in sense_lines:
                text = get_text_from_element(line_el, use_modern=False)
                if text:
                    items.append({
                        'type': 'line',
                        'text': text,
                        'verse_num': vn,
                        'line_index': line_idx,
                    })
                    line_idx += 1
            if items and items[-1]['type'] != 'verse-gap':
                items.append({
                    'type': 'verse-gap',
                    'text': '',
                    'verse_num': '',
                    'line_index': -1,
                })
    return items


# ── ElevenLabs TTS with caching ──

def tts_elevenlabs(text, voice_id=VOICE_ID, model_id=MODEL_ID, settings=None):
    """Call ElevenLabs API for a single line. Uses cache if available."""
    if settings is None:
        settings = VOICE_SETTINGS

    # Check cache first
    key = cache_key(text, voice_id, model_id, settings)
    cached = get_cached_audio(key)
    if cached is not None:
        return cached, True  # True = from cache

    url = f"https://api.elevenlabs.io/v1/text-to-speech/{voice_id}"
    headers = {
        "xi-api-key": ELEVENLABS_API_KEY,
        "Content-Type": "application/json",
        "Accept": "audio/mpeg",
    }
    payload = {
        "text": text,
        "model_id": model_id,
        "voice_settings": settings,
    }
    resp = requests.post(url, json=payload, headers=headers)
    if resp.status_code != 200:
        print(f"  API error {resp.status_code}: {resp.text[:200]}")
        return AudioSegment.silent(duration=500), False

    audio_seg = AudioSegment.from_mp3(io.BytesIO(resp.content))
    save_to_cache(key, audio_seg)
    return audio_seg, False  # False = freshly generated


def generate_silence_seg(duration_ms):
    """Generate silence as an AudioSegment."""
    return AudioSegment.silent(duration=duration_ms)


def stitch_chapter(items, voice_id=VOICE_ID, model_id=MODEL_ID,
                   settings=None, line_pause_ms=LINE_PAUSE_MS,
                   verse_pause_ms=VERSE_PAUSE_MS, stitch_only=False):
    """
    Generate (or re-stitch) a full chapter.
    If stitch_only=True, ONLY uses cached audio — no API calls.
    Returns (AudioSegment, timing_list, stats_dict).
    """
    if settings is None:
        settings = VOICE_SETTINGS

    combined = AudioSegment.empty()
    timing = []
    current_time_ms = 0
    stats = {"cached": 0, "generated": 0, "skipped": 0, "chars_used": 0}

    speakable = [i for i in items if i['type'] == 'line']
    total = len(speakable)
    done = 0

    for item in items:
        if item['type'] == 'verse-gap':
            combined += generate_silence_seg(verse_pause_ms)
            current_time_ms += verse_pause_ms
            continue

        if item['type'] != 'line':
            continue

        done += 1
        key = cache_key(item['text'], voice_id, model_id, settings)

        if stitch_only:
            cached = get_cached_audio(key)
            if cached is None:
                print(f"  SKIP [{done}/{total}] v{item['verse_num']}: not in cache")
                stats["skipped"] += 1
                continue
            audio_seg = cached
            stats["cached"] += 1
            tag = "cache"
        else:
            audio_seg, from_cache = tts_elevenlabs(item['text'], voice_id, model_id, settings)
            if from_cache:
                stats["cached"] += 1
                tag = "cache"
            else:
                stats["generated"] += 1
                stats["chars_used"] += len(item['text'])
                tag = "NEW"
                time.sleep(0.05)

        start_ms = current_time_ms
        combined += audio_seg
        current_time_ms += len(audio_seg)

        timing.append({
            'start': round(start_ms / 1000, 3),
            'end': round(current_time_ms / 1000, 3),
            'type': 'line',
            'text': item['text'],
            'verse': item['verse_num'],
            'lineIndex': item['line_index'],
        })

        print(f"  [{done}/{total}] [{tag}] v{item['verse_num']}: {item['text'][:50]}...")

        # Line pause
        combined += generate_silence_seg(line_pause_ms)
        current_time_ms += line_pause_ms

    return combined, timing, stats


print("Core functions loaded!")
print(f"  Cache dir: {CACHE_DIR}")
print(f"  Voice settings: stability={VOICE_SETTINGS['stability']}, "
      f"similarity={VOICE_SETTINGS['similarity_boost']}, "
      f"style={VOICE_SETTINGS['style']}")
print(f"  Supports: line caching, stitch-only mode, surgical regeneration")

In [ ]:
# Cell 4: Voice test — Samuel the Nigerian
from IPython.display import Audio, display

VOICE_D_ID = "ddDFRErfhdc2asyySOG5"  # Samuel the Nigerian

items = parse_chapter('books/enos.html', 'enos', 1)
speakable = [i for i in items if i['type'] == 'line']
test_items = speakable[:5]

print(f"Voice D test (Samuel): {len(test_items)} lines from Enos 1")
for i, item in enumerate(test_items):
    print(f"  {i+1}. [{item['verse_num']}] {item['text'][:70]}...")

total_chars = sum(len(item['text']) for item in test_items)
print(f"\nChars: ~{total_chars}")

print(f"\n{'='*40}")
print(f"Generating Voice D (Samuel): {VOICE_D_ID}")
print(f"{'='*40}")
combined_d = AudioSegment.empty()
for i, item in enumerate(test_items):
    print(f"  [{i+1}/5] {item['text'][:50]}...")
    seg, cached = tts_elevenlabs(item['text'], voice_id=VOICE_D_ID)
    tag = "cache" if cached else "NEW"
    print(f"         [{tag}]")
    combined_d += seg
    combined_d += generate_silence_seg(LINE_PAUSE_MS)
    if not cached:
        time.sleep(0.1)

combined_d.export("test_voice_D.mp3", format="mp3", bitrate="128k")
print(f"\nVoice D (Samuel): {len(combined_d)/1000:.1f}s")

print(f"\n{'='*40}")
print("VOICE D (Samuel the Nigerian):")
display(Audio("test_voice_D.mp3"))

In [ ]:
# Cell 5: FULL GENERATION — Enos with Samuel
# Per-line caching: re-running is cheap (cached lines skip API)

BOOK_ID = 'enos'
CHAPTER = 1
SAMUEL_VOICE_ID = "ddDFRErfhdc2asyySOG5"  # Samuel

items = parse_chapter('books/enos.html', BOOK_ID, CHAPTER)
speakable = [i for i in items if i['type'] == 'line']
total_chars = sum(len(i['text']) for i in speakable)

print(f"Full Enos generation with Samuel:")
print(f"  {len(speakable)} lines, ~{total_chars} characters")
print(f"  Voice: {SAMUEL_VOICE_ID}")
print(f"  Cached lines reused automatically")
print(f"  Generating...\n")

combined, timing, stats = stitch_chapter(
    items, voice_id=SAMUEL_VOICE_ID
)

# Export MP3
Path('audio').mkdir(exist_ok=True)
mp3_path = f'audio/{BOOK_ID}-{CHAPTER}.mp3'
combined.export(mp3_path, format='mp3', bitrate='128k')

duration_s = len(combined) / 1000
file_size = os.path.getsize(mp3_path)

# Export timing manifest
json_path = f'audio/{BOOK_ID}-{CHAPTER}.json'
manifest = {
    'book': BOOK_ID,
    'bookName': 'Enos',
    'chapter': CHAPTER,
    'voice': {
        'provider': 'elevenlabs',
        'voice_id': SAMUEL_VOICE_ID,
        'model_id': MODEL_ID,
        'settings': VOICE_SETTINGS,
        'name': 'Samuel',
    },
    'pauses': {
        'line_ms': LINE_PAUSE_MS,
        'verse_ms': VERSE_PAUSE_MS,
    },
    'duration': round(duration_s, 3),
    'lines': timing,
}
with open(json_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"\n{'='*50}")
print(f"Audio: {duration_s:.1f}s ({file_size/1024:.0f} KB)")
print(f"Saved: {mp3_path}")
print(f"Saved: {json_path}")
print(f"\nStats:")
print(f"  From cache: {stats['cached']} lines")
print(f"  Newly generated: {stats['generated']} lines")
print(f"  Credits used: ~{stats['chars_used']}")
print(f"{'='*50}")

from IPython.display import Audio, display
display(Audio(mp3_path))

In [ ]:
# Cell 6: Download Samuel's Enos files
from google.colab import files
import shutil

# Copy to distinct filenames
shutil.copy('audio/enos-1.mp3', 'enos-1-Samuel.mp3')
shutil.copy('audio/enos-1.json', 'enos-1-Samuel.json')

files.download('enos-1-Samuel.mp3')
files.download('enos-1-Samuel.json')
print("Downloading Samuel files: enos-1-Samuel.mp3, enos-1-Samuel.json")